# Context Retrieval with Deep Agents: Take-Home

Build a research assistant for any topic you like. The notebook downloads a few Wikipedia pages into a folder, and a deep agent searches them to write you a one-page guide.

You'll try three things from the session:

1. **Search before reading:** grep and semantic search instead of opening every file.
2. **Keep big results out of the main agent:** a subagent does the reading.
3. **Put dates in the text:** so the agent can tell old notes from new ones.

# Setup

Install the packages and connect a model.

In [ ]:
%pip install -q deepagents langchain-anthropic langchain-huggingface sentence-transformers

In [ ]:
# @title Wrap long output lines (run this cell)
# @markdown Wraps long lines in cell output, so you don't have to scroll sideways.
from IPython.display import HTML, display

def set_css(*args):
    display(HTML('''
    <style>
      pre { white-space: pre-wrap; word-break: break-word; }
    </style>
    '''))

get_ipython().events.register('pre_run_cell', set_css)

Add `ANTHROPIC_API_KEY` in Colab's **Secrets** tab (left sidebar: key icon) and turn on notebook access.

In [ ]:
from google.colab import userdata

ANTHROPIC_API_KEY = userdata.get("ANTHROPIC_API_KEY")

# To use an OpenAI key (or another model provider) instead, just switch it:
# OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")

In [ ]:
from langchain.chat_models import init_chat_model

# Haiku keeps this notebook cheap. For better answers, switch to "anthropic:claude-sonnet-5".
model = init_chat_model(
    model="anthropic:claude-haiku-5-5",
    api_key=ANTHROPIC_API_KEY,
)

# To use a different provider instead, replace the block above, for example:
# model = init_chat_model("openai:gpt-6", api_key=OPENAI_API_KEY)

In [ ]:
# @title Helpers (run this cell)
# @markdown Two helpers keep the output short:
# @markdown - `show_steps` prints each tool call, the final answer, and the peak context
# @markdown - `peak_context` returns the most tokens the agent saw at once
from langchain_core.messages import AIMessage, ToolMessage

def peak_context(result):
    ai_messages = [m for m in result["messages"] if isinstance(m, AIMessage)]
    return max(m.usage_metadata["input_tokens"] for m in ai_messages)

def show_steps(result):
    failed = {m.tool_call_id for m in result["messages"] if isinstance(m, ToolMessage) and m.status == "error"}
    for m in result["messages"]:
        if isinstance(m, AIMessage):
            for call in m.tool_calls:
                if call["id"] in failed:
                    continue  # skip calls the model got wrong and retried
                args = str(call["args"])
                print("->", call["name"], args if len(args) < 110 else args[:110] + "...")
    print("\n" + result["messages"][-1].text)
    print(f"\n[Peak context: {peak_context(result):,} tokens]")

# Pick your topic

Change `TOPIC` and `PAGES` to anything on Wikipedia (3 to 5 pages), and write a `SPECIFIC_QUESTION` that one of the pages answers.

In [ ]:
TOPIC = "a weekend in Lisbon"
PAGES = ["Alfama", "Belém Tower", "Sintra", "Portuguese cuisine"]
SPECIFIC_QUESTION = "When was Belém Tower built, and what was it for?"

# Some other ideas:
# TOPIC = "the history of coffee"
# PAGES = ["History of coffee", "Espresso", "Coffeehouse", "Coffea arabica"]
# SPECIFIC_QUESTION = "Who invented the espresso machine, and when?"
#
# TOPIC = "the history of video games"
# PAGES = ["Pong", "Nintendo Entertainment System", "Tetris", "Minecraft"]
# SPECIFIC_QUESTION = "Who created Tetris, and where?"

In [ ]:
import requests
from pathlib import Path

ROOT = Path("library")
ROOT.mkdir(exist_ok=True)

# Download each page's plain text from Wikipedia and save it as a file in library/
for title in PAGES:
    resp = requests.get(
        "https://en.wikipedia.org/w/api.php",
        params={"action": "query", "prop": "extracts", "explaintext": 1,
                "redirects": 1, "format": "json", "titles": title},
        headers={"User-Agent": "context-retrieval-workshop/1.0"},
        timeout=30,
    )
    text = next(iter(resp.json()["query"]["pages"].values())).get("extract", "")
    (ROOT / f"{title.lower().replace(' ', '-')}.md").write_text(f"# {title}\n\n{text}")
    print(f"{title:<30} ~{len(text) // 4:>6,} tokens")

# 1: Search before reading

With `FilesystemBackend`, the agent's file tools work on the `library/` folder. It only sees what's inside, so its paths start at `/`.

Watch the tool calls: the agent should grep first, then read only what matches.

In [ ]:
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend

backend = FilesystemBackend(root_dir=ROOT, virtual_mode=True)

agent = create_deep_agent(
    model=model,
    backend=backend,
)

result = agent.invoke({"messages": [{"role": "user", "content":
    f"{SPECIFIC_QUESTION} Answer from the files in /, and say which file you used."
}]})
show_steps(result)

Compare the peak context with the page sizes above. Searching first means the agent never loads most of them.

## Search by meaning

grep only matches exact words. A semantic search tool matches by meaning. This cell splits the pages into paragraphs, loads them into a vector store, and wraps the search in a tool.

In [ ]:
from langchain_core.documents import Document
from langchain_core.tools import tool
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_huggingface import HuggingFaceEmbeddings

# Split each page into paragraphs (skipping short lines like headings)
paragraphs = [Document(page_content=p, metadata={"source": f"/{path.name}"})
              for path in ROOT.glob("*.md")
              for p in path.read_text().split("\n") if len(p) > 200]
# Store them in a vector store, using a small, free embedding model that runs locally
vector_store = InMemoryVectorStore(HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2"))
vector_store.add_documents(paragraphs)

@tool
def search_library(query: str) -> str:
    """Search the library by meaning. Returns the 4 best-matching paragraphs and the file each came from."""
    docs = vector_store.similarity_search(query, k=4)
    return "\n\n".join(f"[{d.metadata['source']}]\n{d.page_content}" for d in docs)

Ask something using words the pages probably don't. The agent has both grep and `search_library`. Which does it reach for?

In [ ]:
VAGUE_QUESTION = "Where would a local go for a cheap, filling meal?"

agent = create_deep_agent(
    model=model,
    tools=[search_library],
    backend=backend,
)

result = agent.invoke({"messages": [{"role": "user", "content": VAGUE_QUESTION}]})
show_steps(result)

# 2: Keep big results out of the main agent

Here the main agent can't search. It sends questions to a `researcher` subagent, which reads in its own context window and sends back a few bullet points. The main agent only sees those, then writes `/guide.md`.

In [ ]:
researcher = {
    "name": "researcher",
    "description": "Answers one question from the library. Give it the question.",
    "system_prompt": "Answer the question using grep, read_file, and search_library. Reply in 3 short bullet points, each with the file it came from.",
    "tools": [search_library],
}

agent = create_deep_agent(
    model=model,
    subagents=[researcher],
    backend=backend,
    system_prompt="You can't read the library yourself. Send each question to researcher, then write the guide.",
)

result = agent.invoke({"messages": [{"role": "user", "content":
    f"Write a one-page guide to {TOPIC} in /guide.md. Ask researcher 3 questions first."
}]})
show_steps(result)

In [ ]:
print((ROOT / "guide.md").read_text())

# 3: Put dates in the text

These two notes disagree, and only the dates say which one is current. Edit them to fit your topic.

In [ ]:
MY_NOTES = """# My notes

- (2019, from a friend) The rooftop bar near our hotel is open until 2am every night.
- (2026, from its website) The rooftop bar near our hotel now closes at 10pm on weeknights.
"""
(ROOT / "my-notes.md").write_text(MY_NOTES)

agent = create_deep_agent(
    model=model,
    backend=backend,
)

result = agent.invoke({"messages": [{"role": "user", "content":
    "Check /my-notes.md: how late is the rooftop bar open on a Tuesday?"
}]})
show_steps(result)

**Try it:** delete the dates and run the cell again. Without them, the agent can't tell which note is current.

# Your turn

- Make a guide for something you care about. Share your `guide.md`!
- Write a `VAGUE_QUESTION` that grep can't answer but `search_library` can.
- Give `researcher` a different job: a fact-checker, a skeptic, a tour guide with opinions.
- Add a note that contradicts a Wikipedia page. Which one does the agent trust?

# Wrap-up

- **Search before reading:** grep for exact words, semantic search for meaning.
- **Keep big results out of the main agent:** let a subagent do the reading.
- **Put dates in the text:** the model can't use a date it never sees.

To learn more, see the [LangChain Academy Deep Agents course](https://academy.langchain.com/courses/foundation-introduction-to-deepagents).